# Sunrise Coffee Chain — data checks and findings

This notebook backs up every number on the Tableau dashboard and in the README.

* **Part 1** audits the raw DataDNA files against the challenge brief and data dictionary.
* **Part 2** reproduces each finding and recommendation.

Inputs: the five raw challenge CSVs in `data/raw/` (see `data/README.md`) and the four
prepared CSVs from `src/prepare_data.py` in `data/prepared/`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
pd.set_option('display.width', 160)

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
RAW, PREP = ROOT / 'data' / 'raw', ROOT / 'data' / 'prepared'

store = pd.read_csv(RAW / 'dim_store.csv')
date = pd.read_csv(RAW / 'dim_date.csv', parse_dates=['full_date'])
prod = pd.read_csv(RAW / 'dim_product.csv')
daily_raw = pd.read_csv(RAW / 'fact_daily_sales.csv')
monthly = pd.read_csv(RAW / 'fact_store_monthly_profitability.csv')

stores = pd.read_csv(PREP / 'stores.csv')
daily = pd.read_csv(PREP / 'daily_sales.csv')

# monthly facts with store attributes
M = monthly.merge(stores[['Store ID', 'Store', 'Region', 'London vs Rest']],
                  left_on='store_id', right_on='Store ID')
print({name: len(df) for name, df in
       [('dim_store', store), ('dim_date', date), ('dim_product', prod),
        ('fact_daily_sales', daily_raw), ('fact_store_monthly_profitability', monthly)]})

{'dim_store': 60, 'dim_date': 730, 'dim_product': 40, 'fact_daily_sales': 10000, 'fact_store_monthly_profitability': 1440}


## Part 1 — Data audit

### 1.1 Store formats: the brief's three-format comparison isn't possible

In [2]:
print(store.store_format.value_counts().to_string())
print()
print(store.location_type.value_counts().to_string())

store_format
High-Street      58
Transport-Hub     2

location_type
City-Centre             30
Suburban-High-Street    16
Shopping-Centre         12
Train-Station            2


The brief and dictionary promise three formats (High-Street, Retail-Park, Transport-Hub) and ten
location types. The data has **two formats** — 58 High-Street stores and 2 Transport-Hub stores
(both Luton stations) — and four location types. Any format comparison rests on n = 2, so the
dashboard compares regions instead.

### 1.2 `profitability_status` contradicts its own definition

In [3]:
def status_from_margin(m):
    # thresholds from the data dictionary
    return np.select([m > 0.20, m >= 0.08, m >= -0.02, m >= -0.15],
                     ['Highly-Profitable', 'Profitable', 'Break-Even', 'Loss-Making'],
                     'Structural-Loss')

monthly['status_recalc'] = status_from_margin(monthly.net_profit_margin_pct)
mismatch = monthly.profitability_status != monthly.status_recalc
print(f'{mismatch.sum()} of {len(monthly)} rows ({mismatch.mean():.0%}) disagree with the documented thresholds')
pd.crosstab(monthly.profitability_status, monthly.status_recalc,
            rownames=['label in data'], colnames=['label from margin'])

600 of 1440 rows (42%) disagree with the documented thresholds


label from margin,Break-Even,Highly-Profitable,Loss-Making,Profitable,Structural-Loss
label in data,,,,,
Break-Even,176,0,33,159,0
Highly-Profitable,0,327,0,0,0
Loss-Making,0,0,88,0,59
Profitable,0,349,0,232,0
Structural-Loss,0,0,0,0,17


Labels and margins disagree in 42% of rows, and the label ranges overlap (a row labelled
"Profitable" can have a 30% margin). The dashboard ignores the supplied label and recomputes
**Profit status** from net margin with the documented thresholds.

### 1.3 Daily sales: a ~20% sample with duplicate store-days

In [4]:
dup = daily_raw.groupby(['store_id', 'date_id']).size()
print(f'raw rows: {len(daily_raw):,}')
print(f'unique store-days: {len(dup):,}  ->  {len(daily_raw) - len(dup):,} extra rows')
print(f'store-days recorded more than once: {(dup > 1).sum():,}')
print(f'coverage: {len(dup) / (store.store_id.nunique() * date.date_id.nunique()):.1%} of all 60 x 730 store-days')

# the repeats are not exact copies: totals differ between copies of the same store-day
rep = daily_raw[daily_raw.duplicated(['store_id', 'date_id'], keep=False)]
spread = rep.groupby(['store_id', 'date_id']).total_revenue_gbp.agg(lambda s: s.max() - s.min())
print(f'median revenue gap between copies of the same store-day: £{spread.median():,.0f}')

raw rows: 10,000
unique store-days: 8,910  ->  1,090 extra rows
store-days recorded more than once: 1,006
coverage: 20.3% of all 60 x 730 store-days
median revenue gap between copies of the same store-day: £521


Daily rows cover only about a fifth of store-days, and repeated store-days carry conflicting
totals. `prepare_data.py` averages the repeats into one row, and the dashboard only uses daily
data as **averages per store-day** (never sums). Totals and P&L come from the monthly table.

### 1.4 "Top product" is not usable

In [5]:
seasonal = prod[prod.product_name.str.contains('Christmas|Festive|Pumpkin', case=False)][['product_id', 'product_name']]
june = daily_raw.merge(date[['date_id', 'month_number']], on='date_id').query('month_number == 6')
print('Seasonal products recorded as the day\'s top product in June:')
print(june[june.product_id.isin(seasonal.product_id)].merge(seasonal, on='product_id')
      .product_name.value_counts().to_string())

Seasonal products recorded as the day's top product in June:
product_name
Festive Latte           30
Christmas Blend 250g    28
Christmas Mince Pie     23
Pumpkin Spice Latte     22


### 1.5 What does hold: the P&L identities

In [6]:
rebuilt = (monthly.total_monthly_revenue_gbp - monthly.total_monthly_cogs_gbp - monthly.monthly_rent_gbp
           - monthly.monthly_staffing_cost_gbp - monthly.loyalty_redemption_cost_gbp)
split = monthly.loyalty_member_spend_gbp + monthly.non_member_spend_gbp
print('net profit identity, max error (£):', round((rebuilt - monthly.net_profit_gbp).abs().max(), 2))
print('member + non-member = revenue, max error (£):', round((split - monthly.total_monthly_revenue_gbp).abs().max(), 2))

net profit identity, max error (£): 0.0
member + non-member = revenue, max error (£): 0.0


## Part 2 — Findings

### 2.0 Estate headline

In [7]:
rev, npf = monthly.total_monthly_revenue_gbp.sum(), monthly.net_profit_gbp.sum()
print(f'revenue (2 yrs):      £{rev/1e6:,.1f}M')
print(f'net profit (2 yrs):   £{npf/1e6:,.1f}M')
print(f'net margin:           {npf/rev:.1%}')
print(f'sales / store / month £{monthly.total_monthly_revenue_gbp.mean()/1e3:,.1f}K')
print(f'loss-making store-months {(monthly.net_profit_gbp < 0).mean():.1%}')
print(f'loyalty rewards (2 yrs) £{monthly.loyalty_redemption_cost_gbp.sum()/1e6:,.2f}M')

revenue (2 yrs):      £89.7M
net profit (2 yrs):   £16.9M
net margin:           18.8%
sales / store / month £62.3K
loss-making store-months 15.6%
loyalty rewards (2 yrs) £1.73M


### 2.1 Sales are flat across stores, so costs decide profit

In [8]:
by_store = M.groupby('Store').agg(revenue=('total_monthly_revenue_gbp', 'sum'),
                                  cogs=('total_monthly_cogs_gbp', 'sum'),
                                  rent=('monthly_rent_gbp', 'sum'),
                                  labour=('monthly_staffing_cost_gbp', 'sum'),
                                  loyalty=('loyalty_redemption_cost_gbp', 'sum'),
                                  net=('net_profit_gbp', 'sum'))
per_month = by_store.revenue / 24
print(f'avg sales per store-month: £{per_month.min()/1e3:.1f}K to £{per_month.max()/1e3:.1f}K '
      f'(coefficient of variation {per_month.std()/per_month.mean():.0%})')

# Net profit = gross profit - rent - labour - loyalty, so its variance splits into covariances
gp = by_store.revenue - by_store.cogs
parts = pd.Series({'gross profit (sales side)': gp.cov(by_store.net),
                   'rent': -by_store.rent.cov(by_store.net),
                   'labour': -by_store.labour.cov(by_store.net),
                   'loyalty': -by_store.loyalty.cov(by_store.net)}) / by_store.net.var()
print('\nshare of the variance in store net profit explained by each line:')
print(parts.map('{:.0%}'.format).to_string())

pct = by_store[['cogs', 'labour']].div(by_store.revenue, axis=0)
print(f'\nCOGS % of revenue ranges {pct.cogs.min():.1%}–{pct.cogs.max():.1%}; '
      f'labour % ranges {pct.labour.min():.1%}–{pct.labour.max():.1%}')

avg sales per store-month: £56.3K to £71.4K (coefficient of variation 5%)

share of the variance in store net profit explained by each line:
gross profit (sales side)    14%
rent                         23%
labour                       63%
loyalty                       0%

COGS % of revenue ranges 36.4%–39.5%; labour % ranges 15.4%–51.7%


Revenue barely moves between stores. Labour cost explains most of the spread in profit, rent
most of the rest; COGS sits in a narrow band. This is the dashboard's store cost map:
labour % on the x-axis separates the stores, COGS % on the y-axis doesn't.

### 2.2 London: same sales, twice the rent

In [9]:
g = M.groupby('London vs Rest').agg(stores=('store_id', 'nunique'),
                                    revenue=('total_monthly_revenue_gbp', 'sum'),
                                    rent=('monthly_rent_gbp', 'sum'),
                                    net=('net_profit_gbp', 'sum'),
                                    months=('net_profit_gbp', 'size'),
                                    loss_months=('net_profit_gbp', lambda s: (s < 0).sum()))
g['sales / store-month'] = g.revenue / g.months
g['rent / store-month'] = g.rent / g.months
g['rent % revenue'] = g.rent / g.revenue
g['net margin'] = g.net / g.revenue
g['loss-making months'] = g.loss_months / g.months
g[['stores', 'sales / store-month', 'rent / store-month', 'rent % revenue', 'net margin', 'loss-making months']]

,stores,sales / store-month,rent / store-month,rent % revenue,net margin,loss-making months
London vs Rest,,,,,,
London,7,"62,246.264","11,744.122",0.189,0.085,0.321
Rest of UK,53,"62,286.648","5,762.102",0.093,0.202,0.134


In [10]:
store_level = M.groupby('Store').agg(rent=('monthly_rent_gbp', 'mean'), revenue=('total_monthly_revenue_gbp', 'mean'))
print(f'store-level correlation, rent vs sales: r = {store_level.rent.corr(store_level.revenue):.2f}')

lon = g.loc['London']
uk = monthly.net_profit_gbp.sum() / monthly.total_monthly_revenue_gbp.sum()
for cut in [0.2, 0.4]:
    m = (lon.net + lon.rent * cut) / lon.revenue
    print(f'cut London rent by {cut:.0%}: London margin {lon["net margin"]:.1%} -> {m:.1%}, '
          f'extra profit £{lon.rent * cut / 2 / 1e6:.2f}M a year')
need = (uk * lon.revenue - lon.net) / lon.rent
print(f'cut needed to reach the {uk:.1%} UK average: {need:.0%}')
print(f'London rent bill: £{lon.rent / 2 / 1e6:.2f}M a year across {int(lon.stores)} stores')

store-level correlation, rent vs sales: r = 0.15
cut London rent by 20%: London margin 8.5% -> 12.2%, extra profit £0.20M a year
cut London rent by 40%: London margin 8.5% -> 16.0%, extra profit £0.39M a year
cut needed to reach the 18.8% UK average: 55%
London rent bill: £0.99M a year across 7 stores


Rent doesn't buy sales (r ≈ 0.15). A rent cut alone barely closes the gap — London needs
a ~55% cut to reach the UK average — and the whole London rent bill is only about £1M a year.
The practical lever is **where the chain expands next**, not renegotiating existing leases.

### 2.3 Weekend staffing doesn't follow demand

In [11]:
d = daily.groupby('Day Type').agg(revenue=('Daily Revenue', 'mean'), hours=('Staff Hours', 'mean'),
                                  staff=('Staff on Shift', 'mean'))
d['revenue per labour hour'] = d.revenue / d.hours
print(d.round(2).to_string())

drop = 1 - d.loc['Weekend', 'revenue'] / d.loc['Weekday', 'revenue']
share_of_hours = drop * 2 / 7                     # weekends are 2 of 7 days
labour_per_year = monthly.monthly_staffing_cost_gbp.sum() / 2
hours_per_year = daily['Staff Hours'].mean() * 365 * store.store_id.nunique()
print(f'\nweekend sales are {drop:.0%} below weekdays')
print(f'cutting weekend hours by the same {drop:.0%} removes {share_of_hours:.1%} of all hours')
print(f'≈ £{share_of_hours * labour_per_year / 1e6:.2f}M a year '
      f'(labour £{labour_per_year/1e6:.1f}M/yr, implied wage £{labour_per_year / hours_per_year:.1f}/hr)')

ratio = (daily.groupby(['Store ID', 'Day Type'])['Daily Revenue'].mean().unstack()
         .pipe(lambda t: t.Weekend / t.Weekday))
print(f'stores with lower weekend sales: {(ratio < 1).sum()} of {len(ratio)} '
      f'(weekend/weekday ratio {ratio.min():.2f}–{ratio.max():.2f})')

           revenue  hours  staff  revenue per labour hour
Day Type                                                 
Weekday  2,196.400 41.370  5.520                   53.090
Weekend  1,662.710 41.550  5.570                   40.010

weekend sales are 24% below weekdays
cutting weekend hours by the same 24% removes 6.9% of all hours
≈ £0.96M a year (labour £13.9M/yr, implied wage £15.3/hr)
stores with lower weekend sales: 60 of 60 (weekend/weekday ratio 0.69–0.92)


Assumptions: labour cost scales with hours, and the daily table (a ~20% sample) represents the
year. A minimum crew per shift would make the real saving smaller.

### 2.4 Loyalty: more members, no more sales

In [12]:
s = M.groupby('Store').agg(member=('loyalty_member_spend_gbp', 'sum'),
                           revenue=('total_monthly_revenue_gbp', 'sum'),
                           margin_num=('net_profit_gbp', 'sum')).join(
    stores.set_index('Store')['Loyalty Adoption Rate'])
s['member share'] = s.member / s.revenue
s['margin'] = s.margin_num / s.revenue
slope = np.polyfit(s['Loyalty Adoption Rate'], s['member share'], 1)[0]
print(f"adoption vs member revenue share: r = {s['Loyalty Adoption Rate'].corr(s['member share']):.3f}, slope = {slope:.2f}")
print(f"adoption vs store sales:          r = {s['Loyalty Adoption Rate'].corr(s.revenue):.3f}")
print(f"adoption vs net margin:           r = {s['Loyalty Adoption Rate'].corr(s.margin):.3f}")
red = monthly.loyalty_redemption_cost_gbp.sum()
print(f'reward cost: £{red/1e6:.2f}M over two years (£{red/2/1e3:,.0f}K a year, {red/monthly.net_profit_gbp.sum():.1%} of net profit)')

adoption vs member revenue share: r = 0.998, slope = 1.00
adoption vs store sales:          r = 0.003
adoption vs net margin:           r = -0.014
reward cost: £1.73M over two years (£865K a year, 10.3% of net profit)


Member share of revenue equals the adoption rate with a slope of 1: members spend exactly like
non-members, so "member revenue" isn't incremental and the brief's *member revenue − reward cost*
formula overstates the programme's value. Higher adoption doesn't raise store sales or margin.

### 2.5 Mix and seasonality are flat

In [13]:
mix_cols = ['Espresso Revenue', 'Food Revenue', 'Filter Revenue', 'Retail Beans Revenue']
mix = daily[mix_cols].sum() / daily['Daily Revenue'].sum()
print(mix.map('{:.1%}'.format).to_string())

mm = monthly.assign(year=monthly.month_id.str[:4], month=monthly.month_id.str[5:])
pivot = mm.pivot_table(index='month', columns='year', values='total_monthly_revenue_gbp', aggfunc='sum')
print(f"\n2023 vs 2024 monthly revenue pattern: r = {pivot['2023'].corr(pivot['2024']):.2f}")
q = mm.assign(q=((mm.month.astype(int) - 1) // 3 + 1)).groupby('q').total_monthly_revenue_gbp.sum()
print('quarter vs average:', (q / q.mean() - 1).map('{:+.1%}'.format).to_dict())

Espresso Revenue        45.6%
Food Revenue            26.1%
Filter Revenue          19.2%
Retail Beans Revenue     9.0%

2023 vs 2024 monthly revenue pattern: r = -0.35
quarter vs average: {1: '+1.4%', 2: '-0.7%', 3: '-0.8%', 4: '+0.1%'}


### 2.6 Stores to review

In [14]:
review = by_store.assign(margin=by_store.net / by_store.revenue,
                         labour_pct=by_store.labour / by_store.revenue,
                         rent_pct=by_store.rent / by_store.revenue,
                         labour_per_month=by_store.labour / 24,
                         rent_per_month=by_store.rent / 24)
review['loss months'] = M.groupby('Store').net_profit_gbp.apply(lambda s: int((s < 0).sum()))
cols = ['margin', 'loss months', 'labour_pct', 'rent_pct', 'labour_per_month', 'rent_per_month']
print(f"median labour % across stores: {review.labour_pct.median():.1%}")
review.sort_values('margin')[cols].head(5)

median labour % across stores: 30.9%


,margin,loss months,labour_pct,rent_pct,labour_per_month,rent_per_month
Store,,,,,,
South London Corner,-0.037,13,0.517,0.133,"29,781.379","7,670.176"
Canary Wharf Express,0.032,13,0.355,0.218,"19,973.694","12,262.101"
Manchester Collective,0.049,8,0.465,0.081,"26,275.225","4,557.988"
North London Yard,0.051,7,0.340,0.204,"20,439.548","12,292.658"
Canary Wharf Quarter,0.087,8,0.330,0.198,"20,821.081","12,452.040"


**South London Corner** is the weakest store (−3.7% margin, a loss in 13 of 24 months), and its
problem is labour, not rent: staffing takes 52% of sales against a 31% median, while its rent is the
lowest in London. **Canary Wharf Express** is the rent-driven case.